In [430]:
import pandas as pd
import numpy as np

In [539]:
bm_raw = pd.read_csv("/content/drive/MyDrive/Ads Biz-fin/Estimate Working/CY'26/1.Jan'26/14.01/Search/Consumption till 13th/bm_raw_new.csv")
bss_pca = pd.read_csv("/content/drive/MyDrive/Ads Biz-fin/Estimate Working/CY'26/1.Jan'26/14.01/Search/Consumption till 13th/bss_pca_raw.csv")
pca_sellers = pd.read_csv("/content/drive/MyDrive/Ads Biz-fin/Estimate Working/CY'26/1.Jan'26/14.01/Search/Consumption till 13th/PCA for Sellers.csv")
classification = pd.read_excel("/content/drive/MyDrive/Ads Biz-fin/Estimate Working/CY'26/3.Mar'26/display files/classification_mapping.xlsx")
Brand_mapping = pd.read_excel("/content/drive/MyDrive/Ads Biz-fin/Estimate Working/CY'26/3.Mar'26/display files/Brand_mapping.xlsx")


In [432]:
bm_raw.columns

Index(['marketplace', 'campaign_id', 'advertiser_id', 'advertiser_name',
       'BUSINESS_ACCOUNT_ID', 'BUSINESS_ACCOUNT_NAME', 'seller_id',
       'adgroup_id', 'brand', 'commodity_name', 'analytic_vertical',
       'analytic_super_category', 'lifestyle_supercategory', 'Supercategory',
       'BU', 'HL_BU', 'Alpha_MP', 'BMP_Tag', 'Team', 'budget_type',
       'cost_model', 'pacing', 'campaign_type', 'campaign_name', 'status',
       'start_date', 'end_date', 'commodity_id', 'Channel', 'allocated_budget',
       'gross_budget', 'uniqueviewcount', 'actioncount', 'total_cost_x',
       'addtobasketcount', 'viewcount', 'cnt_lid', 'attr_units',
       'attr_revenue', 'overburn_share', 'Actual_spend', 'Budget_adgroup',
       'ROI', 'CTR', 'CVR'],
      dtype='object')

In [433]:
bm_raw = bm_raw[bm_raw['total_cost_x']>5]

In [434]:
# bss_pca.columns

In [435]:
#  pca_sellers.columns

In [436]:
classification.columns

Index(['Super Categories_Tag', 'BU_Tag', 'Wrong Tagging', 'New Tag', 'Type',
       'Adv. Name', 'Tag', 'Remarks', 'Super Categories_HL', 'BU_HL', 'BU_HL2',
       'HL Supercat'],
      dtype='object')

In [437]:
Brand_mapping.columns

Index(['Incorrect Account Name', 'Correct Name', 'BU', 'Concatenate',
       'Advertiser Name', 'Tag', 'Ad Account ID', 'Ad Account Name',
       'Business Account ID', 'Business Account Name', 'Vertical',
       'Current SC', 'New SC'],
      dtype='object')

In [438]:
incorrect_brands = Brand_mapping['Ad Account ID'].unique()

bm_raw['New Alpha/MP'] = np.where(
    bm_raw['advertiser_id'].isin(incorrect_brands) | (bm_raw['Team'] == 'IC'),
    'IC',
    bm_raw['Alpha_MP']
)


In [439]:
bm_raw['New Alpha/MP'].value_counts()

,count
New Alpha/MP,
MP,452982
Alpha,42970
IC,2047


In [440]:
bm_raw['FCFF Alpha/MP'] = np.where(bm_raw['marketplace'] == "HYPERLOCAL","HL",bm_raw['Alpha_MP'])

In [441]:
bm_raw['FCFF Alpha/MP'].value_counts()

,count
FCFF Alpha/MP,
MP,454825
Alpha,36818
HL,6356


In [442]:
brand_map = dict(zip(Brand_mapping['Vertical'], Brand_mapping['New SC']))
sc_map = dict(zip(classification['Wrong Tagging'], classification['New Tag']))

conditions = [
    bm_raw['marketplace'] == "Grocery",
    bm_raw['analytic_vertical'].isin(brand_map.keys()),
    bm_raw['analytic_super_category'].isin(brand_map.keys())
]

choices = [
    "Grocery",
    bm_raw['analytic_vertical'].map(brand_map),
    bm_raw['analytic_super_category'].map(brand_map)
]

fallback = bm_raw['analytic_super_category'].map(sc_map).fillna(bm_raw['analytic_super_category'])

bm_raw['New Supercat'] = np.select(conditions, choices, default=fallback)


In [443]:
# print(bm_raw['New Supercat'].value_counts().to_string())


- Supercategory matching with FCFF's Super category

In [444]:
bm_raw['SC match FCFF'] = bm_raw['New Supercat'].isin(classification['Super Categories_Tag'])

In [445]:
# bm_raw['SC match FCFF'].value_counts()

In [446]:
sc_bu_map = dict(zip(classification['Super Categories_Tag'], classification['BU_Tag']))

bm_raw['New BU'] = bm_raw['New Supercat'].map(sc_bu_map)

In [447]:
print(bm_raw['New BU'].isnull().sum())

289


In [448]:
bm_raw['Spends'] = bm_raw['total_cost_x']

In [449]:
bm_raw['Spends'].sum()

np.float64(1159516729.84)

In [450]:
brand_map = dict(zip(Brand_mapping['Incorrect Account Name'], Brand_mapping['Correct Name']))

bm_raw['lookup_key'] = bm_raw['advertiser_name'].astype(str) + bm_raw['analytic_super_category'].astype(str)

cond_a = bm_raw['lookup_key'].isin(Brand_mapping['Concatenate'])
cond_b = bm_raw['analytic_super_category'].isin(Brand_mapping['BU'])

bm_raw['Brand'] = np.where(
    cond_a & cond_b,
    bm_raw['advertiser_name'].map(brand_map),
    bm_raw['brand']
)
bm_raw['Brand'] = bm_raw['Brand'].fillna(bm_raw['brand'])


# BSS PCA

In [553]:
bss_pca.columns

Index(['Campaign ID', 'Marketplace', 'BU', 'Supercategory', 'HL_BU', 'store',
       'analytic_super_category', 'Brands', 'Store Name',
       'creative_template_id', 'creative_type', 'costmodel', 'Ad Account ID',
       'Ad Account Name', 'Business Account ID', 'Business Account Name',
       'Team', 'Alpha_MP', 'BMP_Tag', 'Channel', 'lifestyle_supercategory',
       'spend', 'views', 'clicks', 'ppv', 'click_direct_units',
       'click_indirect_units', 'click_direct_revenue',
       'click_indirect_revenue', 'Product', 'Average CPC', 'CTR', 'ROI',
       'New Alpha/MP', 'New Supercat', 'SC match FCFF', 'New BU'],
      dtype='object')

In [554]:
bss_pca = bss_pca[bss_pca['spend']>5].copy()

- New Alpha/MP

In [555]:
incorrect_brands = Brand_mapping['Advertiser Name'].unique()

bss_pca['New Alpha/MP'] = np.where(
    bss_pca['Business Account Name'].isin(incorrect_brands) | (bss_pca['Team'] == 'IC'),
    'IC',
    bss_pca['Alpha_MP']
)


- New Supercat

In [557]:
class_map = classification.drop_duplicates('Wrong Tagging').set_index('Wrong Tagging')['New Tag'].to_dict()

conditions = [ bss_pca['BU'] == "Grocery", bss_pca['BU'] == "Mobile", bss_pca['Supercategory'] == "PetCare", bss_pca['Supercategory'].isin(class_map.keys()) ]
choices = [ "Grocery", "Mobile", "Pets", bss_pca['Supercategory'].map(class_map) ]

bss_pca['New Supercat'] = np.select(conditions, choices, default=bss_pca['Supercategory'])


In [559]:
bss_pca['New Supercat'].isnull().sum()

np.int64(0)

- Supercategory matching with FCFF's Super category

In [560]:
bss_pca['SC match FCFF'] = bss_pca['New Supercat'].isin(classification['Super Categories_Tag'])

In [561]:
bss_pca['SC match FCFF'].value_counts()

,count
SC match FCFF,
True,31973
False,468


- New BU

In [563]:
sc_bu_map = classification.drop_duplicates('Super Categories_Tag').set_index('Super Categories_Tag')['BU_Tag'].to_dict()

bss_pca['New BU'] = bss_pca['New Supercat'].map(sc_bu_map)

- Brand

In [565]:
brand_map = Brand_mapping.drop_duplicates('Incorrect Account Name').set_index('Incorrect Account Name')['Correct Name'].to_dict()

bss_pca['lookup_key'] = bss_pca['Ad Account Name'].astype(str) + bss_pca['Supercategory'].astype(str)

cond_a = bss_pca['lookup_key'].isin(Brand_mapping['Concatenate'])
cond_b = bss_pca['Supercategory'].isin(Brand_mapping['BU'])

bss_pca['Brand'] = np.where(
    cond_a & cond_b,
    bss_pca['Ad Account Name'].map(brand_map),
    bss_pca['Brands']
)
bss_pca['Brand'] = bss_pca['Brand'].fillna(bss_pca['Brands'])

- FCFF Alpha/MP

In [566]:
bss_pca['FCFF Alpha/MP'] = np.where(bss_pca['Alpha_MP'] == "IC","Alpha",np.where(bss_pca['Alpha_MP'] =="3P","MP",bss_pca['Alpha_MP']))

In [567]:
bss_pca['BUxBrand'] = bss_pca['New BU'].astype(str) + bss_pca['Brands'].astype(str)
lookup_map = bss_pca.drop_duplicates('BUxBrand').set_index('BUxBrand')['New Supercat'].to_dict()

bss_pca['Check'] = bss_pca['BUxBrand'].map(lookup_map)

In [568]:
bss_pca[bss_pca['New BU'] == 0 ]['spend'].sum()

np.float64(0.0)

- check and redo the Supercat and BU Mapping

In [569]:
bss_pca['New Supercat'] = np.where(bss_pca['SC match FCFF'] == False,bss_pca['Check'], bss_pca['New Supercat'])

In [570]:
sc_bu_map = classification.drop_duplicates('Super Categories_Tag').set_index('Super Categories_Tag')['BU_Tag'].to_dict()

bss_pca['New BU'] = bss_pca['New Supercat'].map(sc_bu_map)

In [571]:
bss_pca['SC match FCFF'] = bss_pca['New Supercat'].isin(classification['Super Categories_Tag'])

# PCA for Sellers

In [469]:
pca_sellers.columns

Index(['campaign_id', 'seller_id', 'BU', 'Supercategory', 'Store Name',
       'analytic_super_category', 'lifestyle_supercategory', 'brand',
       'KAM/NKAM', 'BMP_Tag', 'Channel', 'status', 'adspends', 'views',
       'clicks', 'revenue', 'units', 'CTR', 'CVR', 'ROI'],
      dtype='object')

In [470]:
pca_seller = pca_sellers.copy()

- Alpha/MP

In [471]:
pca_seller['Alpha/MP'] = "MP"

- Campaign Type

In [472]:
pca_seller['Campaign Type'] = "PCA"

- New Supercatagory

In [473]:
sc_map_cls = classification.drop_duplicates('Wrong Tagging').set_index('Wrong Tagging')['New Tag'].to_dict()

pca_seller['New Supercatagory'] = pca_seller['Supercategory'].map(sc_map_cls).fillna(pca_seller['Supercategory'])

- Supercategory matching with FCFF's Super category

In [475]:
pca_seller['SC match FCFF'] = pca_seller['New Supercatagory'].isin(classification['Super Categories_Tag'])

In [476]:
pca_seller['SC match FCFF'].value_counts()

,count
SC match FCFF,
True,298156
False,7707


In [477]:
print(round(pca_seller[pca_seller['SC match FCFF'] == True]['adspends'].sum()))

9361205


- BU

In [478]:
sc_bu_map =classification.drop_duplicates('Super Categories_Tag').set_index('Super Categories_Tag')['BU_Tag'].to_dict()
pca_seller['BU'] = pca_seller['New Supercatagory'].map(sc_bu_map)

In [479]:
#  pca_seller['BU'].value_counts()

In [480]:
brand_sc_map =pca_seller.drop_duplicates('brand').set_index('brand')['Supercategory'].to_dict()
pca_seller['Check'] = pca_seller['brand'].map(brand_sc_map)

- Check and redo Supercat and BU mapping

In [481]:
pca_seller['New Supercatagory'] = np.where(
    (pca_seller['SC match FCFF'] == False) & (pca_seller['New Supercatagory'] != 'Not assigned'),
    pca_seller['Check'],
    pca_seller['New Supercatagory']
)


In [482]:
sc_bu_map =classification.drop_duplicates('Super Categories_Tag').set_index('Super Categories_Tag')['BU_Tag'].to_dict()
pca_seller['BU'] = pca_seller['New Supercatagory'].map(sc_bu_map)

In [484]:
pca_seller['SC match FCFF'] = pca_seller['New Supercatagory'].isin(classification['Super Categories_Tag'])

In [485]:
print(round(pca_seller[pca_seller['SC match FCFF'] == True]['adspends'].sum()))

9457362
